# Character Encodings and Binary Data in Bioinformatics

**Tier 0 -- Computational Foundations | Module 4**

---

## Why This Matters

Every bioinformatics pipeline starts with reading data from files. FASTA sequences, GenBank records, PDB coordinates, GFF annotations -- they all live as bytes on disk. When your Python script reads garbled text, when special characters in gene names vanish, or when a collaborator's CSV from Windows looks wrong on your Linux server, the root cause is almost always an **encoding mismatch**.

This module gives you a deep, practical understanding of character encodings so you can confidently handle any file you encounter.

### Learning Objectives

By the end of this module you will be able to:
- Explain how text is stored as bytes (bits, bytes, hexadecimal)
- Describe ASCII and why it matters for biological sequence data
- Contrast UTF-8, Latin-1, and legacy Cyrillic encodings
- Diagnose and fix common encoding problems in bioinformatics files
- Write robust Python code that handles encodings correctly
- Deal with special characters in gene names, FASTA headers, and database IDs

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook teaches core computational literacy used before any serious bioinformatics analysis: reproducible shell workflows, stats reasoning, and environment hygiene.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Difference between command syntax and conceptual model (what changes state vs what only inspects).
- Interpreting statistical output: p-value alone is not enough without effect size and assumptions.
- Reproducibility: the same command can yield different outputs if input files or working directory differ.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Module 0.3: Bash Scripting for Bioinformatics](../../Tier_0_Computational_Foundations/03_Bash_Scripting/01_bash_scripting.ipynb) | [Next: R Fundamentals for Bioinformatics →](../../Tier_0_Computational_Foundations/05_R_Basics/01_r_fundamentals.ipynb)

---

---

## 1. From Bits to Bytes

Computers store everything as sequences of 0s and 1s. Eight binary digits form one **byte**, which can represent 256 distinct values (0--255).

| Notation | Example | Meaning |
|----------|---------|-------|
| Binary | `01000001` | 8 bits = 1 byte |
| Decimal | `65` | Human-friendly |
| Hexadecimal | `0x41` | Compact (2 hex digits = 1 byte) |

Hexadecimal (base-16) uses digits `0-9` and letters `A-F`. It is the standard way to display raw file bytes because each byte maps to exactly two hex digits.

In [ ]:
# Explore the relationship between decimal, binary, and hex
print(f"{'Decimal':>7}  {'Binary':>10}  {'Hex':>5}  {'Character'}")
print("-" * 40)

for value in [0, 10, 32, 48, 65, 90, 97, 122, 127, 200, 255]:
    binary = format(value, '08b')
    hexadecimal = format(value, '02X')
    # Only show printable ASCII characters
    char = chr(value) if 32 <= value < 127 else '--'
    print(f"{value:7d}  {binary:>10}  0x{hexadecimal}  '{char}'")

In [ ]:
# Python's built-in functions for conversions
print("ord('A') =", ord('A'))        # Character -> integer code point
print("chr(65)  =", chr(65))         # Integer code point -> character
print("hex(65)  =", hex(65))         # Integer -> hex string
print("bin(65)  =", bin(65))         # Integer -> binary string
print("int('41', 16) =", int('41', 16))  # Hex string -> integer

---

## 2. ASCII -- The Universal Foundation

ASCII (American Standard Code for Information Interchange, 1963) defines meanings for byte values 0--127. Every modern encoding is backward-compatible with ASCII.

| Range | Content |
|-------|---------|
| 0--31 | Control characters (newline `\n`=10, tab `\t`=9, carriage return `\r`=13) |
| 32 | Space |
| 48--57 | Digits `0`--`9` |
| 65--90 | Uppercase `A`--`Z` |
| 97--122 | Lowercase `a`--`z` |
| Other | Punctuation, symbols (`>`, `|`, `_`, etc.) |

### Why ASCII matters for bioinformatics

DNA (`ATGCN`), RNA (`AUGC`), and amino acid one-letter codes (`ACDEFGHIKLMNPQRSTVWY`) are all plain ASCII. FASTA headers, FASTQ quality scores, GFF columns -- all ASCII. This means **sequence data itself never has encoding issues**. The problems arise in *annotations*, *author names*, *organism names*, and *free-text fields*.

In [ ]:
# DNA nucleotides are pure ASCII
nucleotides = 'ATGC'
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

print("DNA bases in ASCII:")
for base in nucleotides:
    print(f"  {base} -> code {ord(base)} -> binary {format(ord(base), '08b')}")

print("\nAll amino acid codes are ASCII:")
all_ascii = all(ord(aa) < 128 for aa in amino_acids)
print(f"  {amino_acids} -> all < 128? {all_ascii}")

In [ ]:
# FASTQ quality scores: also ASCII
# Phred+33 encoding maps quality 0-93 to ASCII 33-126
quality_string = "IIIIIIIIIIFFFFFFFDDDDDBBBBBB"

print("FASTQ quality decoding (Phred+33):")
for char in quality_string[:10]:  # Show first 10
    phred = ord(char) - 33
    error_prob = 10 ** (-phred / 10)
    print(f"  '{char}' -> ASCII {ord(char)} -> Phred {phred} -> error rate {error_prob:.6f}")

---

## 3. Beyond ASCII: The Encoding Zoo

Byte values 128--255 are where encodings diverge. Different systems historically assigned different characters to these values.

### 3.1 Latin-1 (ISO 8859-1)

Covers Western European languages. Used in many older databases and some BioPython parsers as a fallback. Key property: **every byte value 0--255 is valid**, so decoding *never fails*. This makes it a useful "safety net" encoding, but it may produce wrong characters.

### 3.2 Legacy Cyrillic Encodings

If you work with Russian-language annotations or databases:
- **CP1251** (Windows-1251): Windows standard for Cyrillic
- **CP866**: DOS Cyrillic
- **KOI8-R**: Unix/Linux Cyrillic (older systems)

These are all single-byte, meaning one byte = one character. But the same byte maps to different characters in each!

### 3.3 UTF-8 -- The Modern Standard

UTF-8 is a variable-length encoding for Unicode:
- ASCII characters (0--127): 1 byte (identical to ASCII)
- Latin/Cyrillic/Greek: 2 bytes
- CJK characters: 3 bytes
- Emoji and rare symbols: 4 bytes

**UTF-8 is the default for virtually all modern bioinformatics tools, databases, and file formats.**

In [ ]:
# Same text, different encodings = different bytes
text = "alpha-helix"
greek_text = "\u03b1-helix"  # Using actual Greek alpha: alpha-helix

print(f"Text: {greek_text}")
print(f"  UTF-8 bytes:   {greek_text.encode('utf-8')}  ({len(greek_text.encode('utf-8'))} bytes)")
print(f"  Latin-1 fails for Greek alpha!")

try:
    greek_text.encode('latin-1')
except UnicodeEncodeError as e:
    print(f"  Error: {e}")

# But plain ASCII text is identical across all encodings
plain = "ATGCGATCGA"
print(f"\n'{plain}' is identical in UTF-8 and Latin-1:")
print(f"  UTF-8:   {plain.encode('utf-8')}")
print(f"  Latin-1: {plain.encode('latin-1')}")
print(f"  Equal?   {plain.encode('utf-8') == plain.encode('latin-1')}")

In [ ]:
# Demonstration: same byte, different interpretation
byte_value = bytes([0xC0])  # Byte 192

print(f"Byte 0xC0 (192) decoded as:")
print(f"  Latin-1: '{byte_value.decode('latin-1')}'  (A with grave accent)")
print(f"  CP1251:  '{byte_value.decode('cp1251')}'   (Cyrillic A)")
print(f"  CP866:   '{byte_value.decode('cp866')}'    (Box-drawing character)")

try:
    byte_value.decode('utf-8')
except UnicodeDecodeError:
    print(f"  UTF-8:   ERROR -- 0xC0 alone is not valid UTF-8")

In [ ]:
# UTF-8 variable-length encoding in action
examples = [
    ('A', 'ASCII letter'),
    ('\u03b1', 'Greek alpha (used in protein structure notation)'),
    ('\u00e9', 'e-acute (common in French author names)'),
    ('\u0410', 'Cyrillic A (Russian annotations)'),
    ('\u4e00', 'CJK character (Chinese/Japanese databases)'),
    ('\U0001F9EC', 'DNA emoji'),
]

print(f"{'Char':<5} {'UTF-8 bytes':<20} {'# bytes':<8} {'Context'}")
print("-" * 70)
for char, context in examples:
    encoded = char.encode('utf-8')
    hex_repr = ' '.join(f'{b:02X}' for b in encoded)
    print(f"{char:<5} {hex_repr:<20} {len(encoded):<8} {context}")

---

## 4. Examining Raw File Bytes

When you suspect an encoding issue, the first step is to look at the raw bytes. Open the file in **binary mode** (`'rb'`) to see exactly what is stored on disk, without any decoding.

In [ ]:
def hexdump(data, width=16):
    """Display bytes in classic hexdump format."""
    for i in range(0, len(data), width):
        chunk = data[i:i + width]
        hex_part = ' '.join(f'{b:02X}' for b in chunk)
        ascii_part = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
        print(f"{i:04X}  {hex_part:<{width * 3}}  {ascii_part}")


# Create a realistic FASTA header and examine its bytes
fasta = ">sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens\n"
fasta += "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP\n"

print("FASTA content:")
print(fasta)
print("\nHexdump:")
hexdump(fasta.encode('utf-8'))

In [ ]:
# What a BOM (Byte Order Mark) looks like
# Some Windows editors prepend UTF-8 BOM: EF BB BF
bom_content = b'\xef\xbb\xbf>sp|P04637|P53_HUMAN\nMEEPQ\n'

print("File with UTF-8 BOM:")
hexdump(bom_content)

# Decoding with utf-8-sig automatically strips the BOM
decoded_with_bom = bom_content.decode('utf-8')
decoded_without_bom = bom_content.decode('utf-8-sig')

print(f"\nWith 'utf-8':     starts with '{decoded_with_bom[0]}' (BOM character, invisible)")
print(f"With 'utf-8-sig': starts with '{decoded_without_bom[0]}' (BOM stripped)")
print(f"\nThe BOM character has code point: U+{ord(decoded_with_bom[0]):04X} (FEFF = ZERO WIDTH NO-BREAK SPACE)")

---

## 5. Common Encoding Problems in Bioinformatics

### 5.1 Windows vs. Unix Line Endings

| System | Line ending | Bytes |
|--------|------------|-------|
| Unix/macOS | `\n` (LF) | `0A` |
| Windows | `\r\n` (CR+LF) | `0D 0A` |
| Old Mac (pre-OS X) | `\r` (CR) | `0D` |

Many bioinformatics tools expect Unix line endings. Windows-style `\r\n` can cause subtle bugs: extra `\r` at the end of sequence lines, off-by-one in coordinates, or FASTA parsers that include `\r` in the sequence.

In [ ]:
# Line ending problems
unix_fasta = b">gene1\nATGCGATCGA\nTTTAAAGGGC\n"
windows_fasta = b">gene1\r\nATGCGATCGA\r\nTTTAAAGGGC\r\n"

print("Unix FASTA (correct):")
hexdump(unix_fasta)

print("\nWindows FASTA (problematic):")
hexdump(windows_fasta)

# Naive parsing breaks with Windows line endings
lines = windows_fasta.decode('ascii').split('\n')
for i, line in enumerate(lines):
    print(f"\nLine {i}: {repr(line)}")
    if not line.startswith('>') and line.strip():
        # The \r at the end will be included in the sequence!
        print(f"  WARNING: sequence has trailing carriage return!")

In [ ]:
# Fix: always strip lines, or use universal newline mode
def parse_fasta_robust(raw_bytes):
    """Parse FASTA from raw bytes, handling any line ending style."""
    # Decode as UTF-8, replace \r\n with \n, then also replace lone \r
    text = raw_bytes.decode('utf-8').replace('\r\n', '\n').replace('\r', '\n')
    
    sequences = {}
    current_header = None
    current_seq = []
    
    for line in text.strip().split('\n'):
        line = line.strip()
        if line.startswith('>'):
            if current_header is not None:
                sequences[current_header] = ''.join(current_seq)
            current_header = line[1:]
            current_seq = []
        elif line:
            current_seq.append(line)
    
    if current_header is not None:
        sequences[current_header] = ''.join(current_seq)
    
    return sequences


# Works with both line ending styles
result = parse_fasta_robust(windows_fasta)
for header, seq in result.items():
    print(f">{header}")
    print(f"{seq}")
    print(f"Length: {len(seq)} (no extra characters)")

### 5.2 Special Characters in Gene Names and Headers

Biological nomenclature is full of non-ASCII characters:

| Character | Usage | Unicode | UTF-8 bytes |
|-----------|-------|---------|-------------|
| alpha | Protein secondary structure (alpha-helix) | U+03B1 | CE B1 |
| beta | Beta-sheet, beta-globin | U+03B2 | CE B2 |
| delta | Delta notation (delta-F508 in CFTR) | U+0394 | CE 94 |
| micro | Microgram, microliter | U+00B5 | C2 B5 |
| degree sign | Melting temperature (Tm = 65 degrees C) | U+00B0 | C2 B0 |
| plus-minus | Error ranges | U+00B1 | C2 B1 |

Problems occur when:
1. A database stores names in UTF-8 but your script reads with Latin-1
2. Copy-paste from Word/PDF introduces invisible Unicode characters
3. Different systems use different representations (e.g., `micro sign` U+00B5 vs. `Greek small mu` U+03BC)

In [ ]:
# Real-world: special characters in biological context
bio_texts = [
    "\u0394F508 mutation in CFTR",           # Delta-F508
    "\u03b1-helix / \u03b2-sheet",           # alpha-helix / beta-sheet
    "50 \u00b5g/\u00b5L concentration",      # micrograms/microliters
    "Tm = 65\u00b0C \u00b1 2\u00b0C",       # degrees, plus-minus
    "p53\u2013mediated apoptosis",            # en-dash (common from Word)
    "Na\u207a/K\u207a ATPase",               # superscript plus
]

for text in bio_texts:
    utf8_bytes = text.encode('utf-8')
    print(f"{text}")
    print(f"  UTF-8: {len(utf8_bytes)} bytes")
    print()

In [ ]:
# The "mojibake" problem: reading UTF-8 data as Latin-1
original = "\u03b1-helix structure"  # alpha-helix structure
utf8_bytes = original.encode('utf-8')

# Correct decoding
correct = utf8_bytes.decode('utf-8')
print(f"Correct (UTF-8):  {correct}")

# Wrong decoding -- this is what "mojibake" looks like
wrong = utf8_bytes.decode('latin-1')  # Latin-1 never fails, but gives garbage
print(f"Wrong (Latin-1):  {wrong}")

# Fixing mojibake: re-encode the wrongly-decoded text, then decode correctly
fixed = wrong.encode('latin-1').decode('utf-8')
print(f"Fixed:            {fixed}")

### 5.3 Invisible Unicode Characters

Copy-pasting from PDFs, Word documents, or web pages often introduces invisible characters that wreak havoc on bioinformatics pipelines:

- **Non-breaking space** (U+00A0): Looks like a space, but is not `' '`
- **Zero-width space** (U+200B): Completely invisible
- **Soft hyphen** (U+00AD): May or may not display
- **Right-to-left mark** (U+200F): Can reverse text direction

In [ ]:
# Invisible characters lurking in copy-pasted sequences
clean_seq = "ATGCGATCGA"
dirty_seq = "ATG\u200BCGA\u00A0TCGA"  # Zero-width space + non-breaking space

print(f"Clean: '{clean_seq}' (length {len(clean_seq)})")
print(f"Dirty: '{dirty_seq}' (length {len(dirty_seq)})")
print(f"They look similar but are NOT equal: {clean_seq == dirty_seq}")
print()

# Reveal the hidden characters
print("Character-by-character inspection of dirty sequence:")
for i, char in enumerate(dirty_seq):
    code = ord(char)
    name = f"U+{code:04X}"
    is_dna = char in 'ATGCNatgcn'
    status = 'DNA' if is_dna else 'INTRUDER'
    print(f"  [{i}] '{char}' ({name}) -> {status}")

In [ ]:
import unicodedata

def sanitize_sequence(seq, valid_chars='ATGCNatgcn'):
    """Remove any character that is not a valid sequence letter.
    
    Reports what was removed so you know if something was wrong.
    """
    valid_set = set(valid_chars)
    cleaned = []
    removed = []
    
    for char in seq:
        if char in valid_set:
            cleaned.append(char)
        elif char not in ('\n', '\r', ' ', '\t'):  # Whitespace is expected
            name = unicodedata.name(char, f'U+{ord(char):04X}')
            removed.append(f"'{char}' ({name})")
    
    if removed:
        print(f"WARNING: Removed {len(removed)} unexpected characters: {', '.join(removed)}")
    
    return ''.join(cleaned)


# Test it
messy_input = "ATG\u200BCGA\u00A0TCGA\r\nGGG\u00ADTTT"
clean = sanitize_sequence(messy_input)
print(f"Result: {clean}")

---

## 6. Best Practices: Reading and Writing Files

### The Golden Rules

1. **Always specify `encoding='utf-8'`** when opening text files
2. **Use `'rb'` mode** when you need to inspect raw bytes or handle binary formats (BAM, gzip)
3. **Use `errors='replace'` or `errors='ignore'`** as a last resort for damaged files
4. **Normalize line endings** early in your pipeline
5. **Validate input** -- strip unexpected characters from sequences

In [ ]:
import tempfile
import os

# GOOD: always specify encoding
tmpfile = os.path.join(tempfile.gettempdir(), 'test_sequence.fasta')

# Writing with explicit encoding
with open(tmpfile, 'w', encoding='utf-8') as f:
    f.write(">sp|P04637|P53_HUMAN Cellular tumor antigen p53\n")
    f.write("MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDD\n")

# Reading with explicit encoding
with open(tmpfile, 'r', encoding='utf-8') as f:
    content = f.read()
    print(content)

os.remove(tmpfile)

In [ ]:
# Robust file reader that tries multiple encodings
def read_text_file(filepath):
    """Read a text file, trying common bioinformatics encodings in order.
    
    Priority:
    1. UTF-8 (modern standard)
    2. UTF-8 with BOM (Windows editors)
    3. Latin-1 (never fails -- fallback, but may give wrong characters)
    """
    encodings = ['utf-8-sig', 'utf-8', 'latin-1']
    
    for encoding in encodings:
        try:
            with open(filepath, 'r', encoding=encoding) as f:
                content = f.read()
            if encoding == 'latin-1':
                print(f"WARNING: Fell back to Latin-1 decoding for {filepath}")
                print("  Some characters may be incorrect. Consider converting to UTF-8.")
            return content
        except UnicodeDecodeError:
            continue
    
    raise ValueError(f"Could not decode {filepath} with any known encoding")


# Demonstration with a temp file
tmpfile = os.path.join(tempfile.gettempdir(), 'test_encoding.txt')

# Write a file with BOM (simulating Windows Notepad)
with open(tmpfile, 'wb') as f:
    f.write(b'\xef\xbb\xbf')  # UTF-8 BOM
    f.write("Gene: TP53 (tumor protein p53)\n".encode('utf-8'))

content = read_text_file(tmpfile)
print(f"Content starts with: '{content[:10]}' (BOM automatically stripped)")

os.remove(tmpfile)

---

## 7. Encoding Detection

When you receive a file with unknown encoding, you can try to detect it. The `chardet` library uses statistical analysis of byte patterns to guess the encoding. It is not perfect, but works well in practice.

In [ ]:
# chardet is available in most bioinformatics environments
# Install with: pip install chardet
try:
    import chardet
    
    # Test with different encodings
    test_texts = {
        'UTF-8': "Gene expression analysis of \u03b1-helix proteins".encode('utf-8'),
        'CP1251': "Анализ экспрессии генов".encode('cp1251'),
        'Latin-1': "R\xe9sum\xe9 des r\xe9sultats".encode('latin-1'),
    }
    
    for name, raw_bytes in test_texts.items():
        detected = chardet.detect(raw_bytes)
        print(f"Actual: {name:<10} -> Detected: {detected['encoding']:<10} "
              f"(confidence: {detected['confidence']:.0%})")

except ImportError:
    print("chardet not installed. Install with: pip install chardet")
    print("\nManual detection approach:")
    print("1. Try UTF-8 first (most common)")
    print("2. If it fails, check if all bytes are < 128 (pure ASCII)")
    print("3. Look for encoding hints in file headers or metadata")
    print("4. Fall back to Latin-1 (always succeeds)")

---

## 8. Database Identifiers and URLs

Bioinformatics database identifiers (UniProt, NCBI, PDB) are always pure ASCII. But when identifiers appear in URLs or filenames, certain characters need **percent-encoding** (also called URL encoding).

In [ ]:
from urllib.parse import quote, unquote

# Gene names with special characters in URLs
gene_queries = [
    "TP53",                     # Simple -- no encoding needed
    "HLA-A*02:01",              # HLA allele with * and :
    "C/EBP\u03b1",              # Transcription factor with Greek letter
    "Na+/K+ ATPase",            # Ion channel with + and spaces
]

base_url = "https://www.uniprot.org/uniprot/?query="

print(f"{'Gene Name':<20} {'URL-encoded query'}")
print("-" * 65)
for gene in gene_queries:
    encoded = quote(gene, safe='')
    print(f"{gene:<20} {base_url}{encoded}")

In [ ]:
# Decoding percent-encoded strings back
encoded_url = "https://www.ncbi.nlm.nih.gov/gene/?term=C%2FEBP%CE%B1"
decoded = unquote(encoded_url)
print(f"Encoded: {encoded_url}")
print(f"Decoded: {decoded}")

---

## 9. Converting Between Encodings

A common task: you receive a file in an old encoding and need to convert it to UTF-8 for your pipeline.

In [ ]:
def convert_file_encoding(input_path, output_path, from_encoding, to_encoding='utf-8'):
    """Convert a text file from one encoding to another."""
    with open(input_path, 'r', encoding=from_encoding) as f_in:
        content = f_in.read()
    
    with open(output_path, 'w', encoding=to_encoding) as f_out:
        f_out.write(content)
    
    in_size = os.path.getsize(input_path)
    out_size = os.path.getsize(output_path)
    print(f"Converted {input_path}")
    print(f"  {from_encoding} ({in_size} bytes) -> {to_encoding} ({out_size} bytes)")


# Demonstration: CP1251 (Cyrillic) to UTF-8
import tempfile, os

# Create a sample CP1251 file
cp1251_file = os.path.join(tempfile.gettempdir(), 'cyrillic_sample.txt')
utf8_file = os.path.join(tempfile.gettempdir(), 'utf8_sample.txt')

with open(cp1251_file, 'w', encoding='cp1251') as f:
    f.write("Homo sapiens - Organism description\n")

convert_file_encoding(cp1251_file, utf8_file, 'cp1251', 'utf-8')

# Verify
with open(utf8_file, 'r', encoding='utf-8') as f:
    print(f"Verification: {f.read().strip()}")

os.remove(cp1251_file)
os.remove(utf8_file)

---

## 10. Practical Summary: Encoding Cheat Sheet

| Scenario | Solution |
|----------|----------|
| Reading any text file | `open(f, encoding='utf-8')` |
| File from Windows with BOM | `open(f, encoding='utf-8-sig')` |
| File has Windows line endings | `text.replace('\r\n', '\n')` or `open(f, newline='')` |
| Garbled characters (mojibake) | Re-encode to bytes with the wrong encoding, decode with the right one |
| Unknown encoding | Use `chardet.detect()` or try UTF-8 then Latin-1 |
| Need to inspect raw bytes | `open(f, 'rb')` then `hexdump()` |
| Sequence has invisible chars | `sanitize_sequence()` with a whitelist |
| Special chars in URLs | `urllib.parse.quote()` / `unquote()` |
| Converting old files | Read with old encoding, write with `encoding='utf-8'` |

### Key Takeaways

1. **Sequence data (ATGC, amino acids) is always ASCII** -- encoding issues only affect annotations and metadata
2. **UTF-8 is the standard** for modern bioinformatics; always use it
3. **Latin-1 never fails to decode** but may give wrong characters -- useful as a fallback, dangerous as a default
4. **When in doubt, hexdump** -- looking at raw bytes resolves all encoding mysteries
5. **Validate and sanitize** -- never trust that input files are well-formed

---

## Exercises

### Exercise 1: Byte Inspector

Write a function `inspect_string(text)` that, for each character in the input:
- Prints the character, its Unicode code point, its UTF-8 byte representation, and the number of bytes
- Flags any non-ASCII character (code point > 127)

Test it with: `"BRCA1 \u0394F508 Tm=65\u00b0C"`

In [ ]:
# YOUR CODE HERE


### Exercise 2: FASTA Encoding Fixer

Write a function `fix_fasta(raw_bytes)` that:
1. Detects and removes any BOM
2. Normalizes line endings to `\n`
3. Removes any non-ASCII characters from sequence lines (but preserves them in headers)
4. Returns a clean string

Test it with:
```python
test_input = b'\xef\xbb\xbf>sp|P04637|P53 \xce\xb1-domain\r\nMEEP\xc2\xa0QSDP\r\nSVEP\r\n'
```

In [ ]:
# YOUR CODE HERE


### Exercise 3: Encoding Detective

The following bytes were extracted from a file. Determine what the text says by trying to decode with different encodings. Which encoding produces readable text?

```python
mystery_bytes = bytes([0xC3, 0xA9, 0x70, 0x69, 0x67, 0xC3, 0xA9, 0x6E, 0xC3, 0xA9, 0x74, 0x69, 0x71, 0x75, 0x65])
```

In [ ]:
# YOUR CODE HERE


---

## Exercise Solutions

In [ ]:
# Solution 1: Byte Inspector
def inspect_string(text):
    print(f"{'Char':<6} {'Code Point':<12} {'UTF-8 Hex':<15} {'Bytes':<6} {'Note'}")
    print("-" * 55)
    for char in text:
        cp = ord(char)
        utf8 = char.encode('utf-8')
        hex_repr = ' '.join(f'{b:02X}' for b in utf8)
        note = 'NON-ASCII' if cp > 127 else ''
        display = char if cp >= 32 else repr(char)
        print(f"{display:<6} U+{cp:04X}       {hex_repr:<15} {len(utf8):<6} {note}")


inspect_string("BRCA1 \u0394F508 Tm=65\u00b0C")

In [ ]:
# Solution 2: FASTA Encoding Fixer
def fix_fasta(raw_bytes):
    # 1. Decode, stripping BOM with utf-8-sig
    text = raw_bytes.decode('utf-8-sig')
    
    # 2. Normalize line endings
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    
    # 3. Clean sequence lines (keep headers intact)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if line.startswith('>'):
            cleaned_lines.append(line)  # Preserve header as-is
        else:
            # Remove non-ASCII from sequence lines
            clean = ''.join(c for c in line if ord(c) < 128 and c not in '\t ')
            cleaned_lines.append(clean)
    
    return '\n'.join(cleaned_lines)


test_input = b'\xef\xbb\xbf>sp|P04637|P53 \xce\xb1-domain\r\nMEEP\xc2\xa0QSDP\r\nSVEP\r\n'
result = fix_fasta(test_input)
print(result)
print()
print("Header preserved Greek alpha:", '\u03b1' in result.split('\n')[0])
print("Sequence cleaned:", result.split('\n')[1])  # Should be MEEPQSDP without non-breaking space

In [ ]:
# Solution 3: Encoding Detective
mystery_bytes = bytes([0xC3, 0xA9, 0x70, 0x69, 0x67, 0xC3, 0xA9, 0x6E, 0xC3, 0xA9, 0x74, 0x69, 0x71, 0x75, 0x65])

for enc in ['utf-8', 'latin-1', 'cp1251', 'ascii']:
    try:
        decoded = mystery_bytes.decode(enc)
        print(f"{enc:>10}: {decoded}")
    except (UnicodeDecodeError, LookupError) as e:
        print(f"{enc:>10}: FAILED - {e}")

print("\nAnswer: The text is 'epigenetique' (French for 'epigenetic'), encoded in UTF-8.")
print("Latin-1 decodes without error but produces garbled accented characters.")

---

[← Previous: Module 0.3: Bash Scripting for Bioinformatics](../../Tier_0_Computational_Foundations/03_Bash_Scripting/01_bash_scripting.ipynb) | [Next: R Fundamentals for Bioinformatics →](../../Tier_0_Computational_Foundations/05_R_Basics/01_r_fundamentals.ipynb)